In [70]:
import json
import os
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse, parse_qs

In [71]:
start_date = "09/03/2025"
end_date = "30/10/2025"
dest_folder_path = "/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/"
output_path = "/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/09_03_2025/"
name = "dest"
devided_by = 5

In [72]:
def split_data(data_to_split:list, by:int) -> list:
    return [data_to_split[i:i + by] for i in range(0, len(data_to_split), by)]

In [73]:
def normalize_urls(urls:list, dates:list) -> list:
        updated_urls = []
        for date in dates:
            for url in urls:
                url_params = list(parse_qs(urlparse(url).query).keys())
                if 'c' in url_params and 'hl' in url_params:
                    return url + f"&arrival={date}"
                else:
                    if 'c' not in url_params:
                        url += '&c=EUR'
                    if 'hl' not in url_params:
                        url += '&hl=fr_CH'
                    updated_urls.append(url + f"&arrival={date}")

        return updated_urls

In [74]:
def get_saturdays(start_date, end_date):
    saturdays = pd.bdate_range(start=start_date, end=end_date, freq='C', weekmask='Sat').strftime('%Y-%m-%d').tolist()
    return saturdays

In [75]:
def get_json_file_content(json_file_path:str, key:str=None) -> object:
    """get json file content
    Args:
        json_file_path (str): json file path
    Returns:
        object: json file content
    """
    if Path(json_file_path).exists():
        with open(json_file_path, 'r') as openfile:
            file_content = json.load(openfile)
            if file_content and key:
                try:
                    return file_content.get(key)
                except KeyError as e:
                    print('error',f"{e}")
            return file_content
    print('error', 'file does not found')

In [76]:
def combine_file_content(file_path:str, file_type:str) -> list:
    match file_type:
        case 'json':
            files = os.listdir(file_path)
            files = list(filter(lambda f: f.endswith('.json'), files))
            print(files)
            files = [f"{file_path + x}" for x in files]
            file_contents = []
            for file in files:
                print(file)
                file_content = get_json_file_content(f"{file}")
                file_contents += file_content
            return file_contents
        case 'csv':
            pass

In [77]:
dest_data = combine_file_content(dest_folder_path, 'json')

['edomizil_17_02_2025.json', 'edomizil_03_03_2025.json', 'edomizil_10_02_2025.json', 'edomizil_24_02_2025.json']
/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/edomizil_17_02_2025.json
/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/edomizil_03_03_2025.json
/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/edomizil_10_02_2025.json
/home/keller/Documents/Jobdev/G2A/old_programs/edomizil/dests/edomizil_24_02_2025.json


In [78]:
len(dest_data) * len(get_saturdays(start_date, end_date)) // devided_by

5080

In [79]:
print(len(dest_data))
cleaned_urls = list(set(dest_data))
cleaned_urls

3175


['https://www.e-domizil.ch/rental/5f53c56786b03585?location=5460aeabb3b30&pricetype=totalPrice&duration=7&timestamp=2025-02-10T11%3A43%3A21%2B01%3A00&id=5f53c56786b03585&searchId=1adbef2678a920c0&screen=search&isHotel=0&clickId=BJ3ZJKM9JQ5NC8XH&sT=dateless&prodName=JM&prodSource=Search',
 'https://www.e-domizil.ch/rental/32460db4d6f3f0b2?location=5460aeabb3b30&pricetype=totalPrice&duration=7&timestamp=2025-02-10T09%3A31%3A52%2B01%3A00&id=32460db4d6f3f0b2&searchId=1adbef2678a920c0&screen=search&isHotel=0&clickId=68GPYNC4PMDH85YW&sT=dateless&prodName=JM&prodSource=Search',
 'https://www.e-domizil.ch/rental/fc37b5ef0f832215189143d4a7d5dafc?location=5460aeabb3b30&pricetype=totalPrice&duration=7&timestamp=2025-02-17T11%3A07%3A57%2B01%3A00&id=fc37b5ef0f832215189143d4a7d5dafc&searchId=eefe92a60118a773&screen=search&isHotel=0&clickId=4TJNK4YM1WMN76MZ&sT=dateless&prodName=JM&prodSource=Search',
 'https://www.e-domizil.ch/rental/889e88fcbb4fb10b77b2c0d247e45c1c?location=5460aeabb3b30&pricetype=t

In [80]:
number_data_per_file = len(cleaned_urls) // devided_by

In [ ]:
splited_datas = split_data(cleaned_urls, number_data_per_file)
splited_datas

In [82]:
saturdays = get_saturdays(start_date, end_date)
print(saturdays)

['2025-09-06', '2025-09-13', '2025-09-20', '2025-09-27', '2025-10-04', '2025-10-11', '2025-10-18', '2025-10-25']


In [85]:
new_splited_datas = []
for data in splited_datas:
    new_splited_datas.append(normalize_urls(data, saturdays))

In [86]:
for i in range(len(new_splited_datas)):
    with open(f"{output_path}{name}_{i+1}.json", 'w') as outfile:
        json.dump(new_splited_datas[i], outfile)